# Scrapers for MyDramaList

You will follow the instructions in Part 4 of Week 2 Tasks. In the top part of the notebook summarize through a table of content what you decided to do and then explain why.

**Table of Contents**

1. [Scraping an aggregation page (top rated movies)](#sec1)
2. [Scraping a dedicated page](#sec2)
3. [Use pagination to scrape multiple pages](#sec3)
4. [Use scrolling to scrape comments](#sec4)

**Why I decided to do these tasks**
<p>I chose these tasks because I have done some scraping before, but I have never scraped a page with dynamic content so it will be a challenge.</p>
<p>For the first two tasks, I will simply use requests and BeautifulSoup to extract the page content. For the final two tasks, I will use Selenium to work with the dynamic content.</p>

<h2>Import requirements</h2>

In [75]:
import requests
from bs4 import BeautifulSoup
import re
import json

<h2>Define common functions</h2>

In [2]:
def fetch_page_content(url):
    """Fetches HTML content from a URL and checks status code."""
    response = requests.get(url)
    if response.status_code == 200:
        return response.text
    else:
        return None

<h2 id="sec1">1. Scraping an aggregation page (top rated movies)</h2>

<h3>Get page content and identify movie box cards</h3>

In [23]:
url = "https://mydramalist.com/movies/top"

html_content = fetch_page_content(url)

soup = BeautifulSoup(html_content, "html.parser")
movie_cards = soup.find_all(class_="box-body")

In [27]:
print(len(movie_cards), "\n")
print(movie_cards[0])

22 

<div class="box-body">
<div class="row">
<div class="col-xs-3 row-cell film-cover cover">
<div class="item">
<a class="block" href="/30499-the-youthful-you-who-was-so-beautiful">
<img alt="Better Days" class="img-responsive cover lazy" data-src="https://i.mydramalist.com/rmEw2s.jpg?v=1"/>
</a>
</div>
</div>
<div class="col-xs-9 row-cell content">
<div class="ranking pull-right"><span>#1</span></div>
<h6 class="text-primary title"><a href="/30499-the-youthful-you-who-was-so-beautiful">Better Days</a>
<a class="btn simple btn-manage-list" data-id="30499" data-stats="mylist:30499" rel="nofollow"><span><i class="far fa-plus"></i></span></a>
</h6>
<span class="text-muted">Chinese Movie - 2019</span>
<p><span class="rating"><span class="fill" style="width:92%"></span></span> <span class="p-l-xs score">9.2</span></p>
<p>The film revolves around a girl who is being bullied at school and her relationship with a tough street kid, with whom she is implicated in the murder of a teenage girl. 

<h3>Extract individual movie content</h3>

<p>Define helpful function</p>

In [93]:
def get_single_movie_info(movie):
    movie_info = {}

    #save movie rank (#1, #2, etc..)
    movie_info["rank"] = movie.find("div", class_="ranking pull-right").get_text(strip=True)[1:]

    #save movie title
    movie_info["title"] = movie.find("h6").get_text(strip=True)

    movie_type_and_year = movie.find("span", class_="text-muted").get_text(strip=True)
    #save movie type (korean movie, chinese movie, etc...)
    pattern = re.compile(r'^[^-]+(?=-)') #get text before the seperator (-)
    movie_info["type"] = pattern.findall(movie_type_and_year)[0].strip()
    #save movie year
    movie_info["release_year"] = movie_type_and_year.split()[-1]

    #save movie rating
    movie_info["rating"] = movie.find("span", class_="p-l-xs score").get_text(strip=True)

    #save movie streaming
    movie_info["streaming"] = False
    movie_streaming = movie.find_all("a", class_="btn-watch-online btn-sm btn white")
    if movie_streaming:
        movie_info["streaming"] = True

    #save movie synopsis (however much fits /is shown on the page)
    movie_synopsis = movie.find_all("p")[-1]
    movie_info["synopsis"] = movie_synopsis.get_text(strip=True)

    #save movie cover image link
    movie_cover_image = movie.find_all("img")[0]
    movie_info["cover_image"] = movie_cover_image["data-src"]
    
    return movie_info

<p>Iterate through movies to save their info</p>

In [94]:
all_movies_info = []

for movie in movie_cards:
    try:
        movie_info = get_single_movie_info(movie)
        all_movies_info.append(movie_info)
    except Exception as e:
        pass

print(all_movies_info[0])

{'rank': '1', 'title': 'Better Days', 'type': 'Chinese Movie', 'release_year': '2019', 'rating': '9.2', 'streaming': True, 'synopsis': 'The film revolves around a girl who is being bullied at school and her relationship with a tough street kid, with whom she is implicated in the murder of a teenage girl. (Source: ScreenDaily) ~~ Adapted from the web…', 'cover_image': 'https://i.mydramalist.com/rmEw2s.jpg?v=1'}


<p>Save to JSON file</p>

In [95]:
with open("top_rated_movies.json", mode="w", encoding="utf-8") as write_file:
    json.dump(all_movies_info, write_file, indent = 2)